In [0]:
from pyspark.sql import functions as F


# ---------------------------------------------
# 1. 현재 Reporting Year / Cutoff Month
# ---------------------------------------------

reporting_df = spark.table(
    "finance.silver.reporting_period"
)

reporting_row = (
    reporting_df
    .orderBy(F.desc("fiscal_year"))
    .first()
)

reporting_year = reporting_row["fiscal_year"]
cutoff_month = reporting_row["cutoff_month"]


print(f"Reporting Year : {reporting_year}")
print(f"Cutoff Month   : {cutoff_month}")


# ---------------------------------------------
# 2. 현재 Actual YTD
# ---------------------------------------------

actual_df = (
    spark.table(
        "finance.silver.fact_actual_monthly_all"
    )

    .filter(
        (F.col("fiscal_year") == reporting_year)
        &
        (F.col("fiscal_month") <= cutoff_month)
    )

    .groupBy(
        "cost_center",
        "department_name"
    )

    .agg(
        F.sum(
            F.col("actual_amount").cast("double")
        ).alias("actual_ytd")
    )
)


# ---------------------------------------------
# 3. 현재 CF05 Budget
# ---------------------------------------------

budget_df = (
    spark.table(
        "finance.silver.fact_plan_version"
    )

    .filter(
        (F.col("fiscal_year") == reporting_year)
        &
        (F.col("plan_version") == "CF05")
    )

    .groupBy(
        "cost_center"
    )

    .agg(
        F.sum(
            F.col("annual_plan_amount").cast("double")
        ).alias("current_budget")
    )
)


# ---------------------------------------------
# 4. Historical 8월 평균 YTD Share
# ---------------------------------------------

historical_df = (
    spark.table(
        "finance.gold.finance_historical_patterns"
    )

    .filter(
        (F.col("fiscal_year") < reporting_year)
        &
        (F.col("fiscal_month") == cutoff_month)
    )

    .groupBy(
        "fiscal_year",
        "cost_center",
        "department_name"
    )

    .agg(
        F.sum(
            F.col("actual_ytd").cast("double")
        ).alias("historical_ytd"),

        F.sum(
            F.col("full_year_actual").cast("double")
        ).alias("historical_full_year")
    )

    .withColumn(
        "ytd_share",

        F.when(
            F.col("historical_full_year") > 0,

            F.col("historical_ytd")
            / F.col("historical_full_year")
        )
    )

    .groupBy(
        "cost_center",
        "department_name"
    )

    .agg(
        F.avg(
            "ytd_share"
        ).alias("avg_historical_ytd_share")
    )
)


# ---------------------------------------------
# 5. Baseline Scenario 데이터 생성
# ---------------------------------------------

scenario_base_df = (
    actual_df

    .join(
        budget_df,
        on="cost_center",
        how="left"
    )

    .join(
        historical_df,
        on=[
            "cost_center",
            "department_name"
        ],
        how="left"
    )

    # Historical Pattern 기반 예상 연말 Actual
    .withColumn(
        "seasonal_eac",

        F.when(
            F.col("avg_historical_ytd_share") > 0,

            F.col("actual_ytd")
            / F.col("avg_historical_ytd_share")
        )
    )

    # 예상 연말 Actual - 현재 Budget
    .withColumn(
        "forecast_gap",

        F.col("seasonal_eac")
        - F.col("current_budget")
    )

    .withColumn(
        "forecast_status",

        F.when(
            F.col("forecast_gap") > 0,
            "PROJECTED_OVERRUN"
        ).otherwise(
            "WITHIN_BUDGET"
        )
    )
)


display(
    scenario_base_df
    .orderBy(
        F.desc("forecast_gap")
    )
)

Reporting Year : 2026
Cutoff Month   : 8


cost_center,department_name,actual_ytd,current_budget,avg_historical_ytd_share,seasonal_eac,forecast_gap,forecast_status
410500,마케팅팀 3,5.55139231E8,7.8911233E8,0.5675934612074448,9.780578335399586E8,1.889455035399586E8,PROJECTED_OVERRUN
410290,국내영업팀 2,6.90156513E8,9.86795667E8,0.6647462740709732,1.0382254702586175E9,5.142980325861752E7,PROJECTED_OVERRUN
410470,품질관리팀 3,7.66505908E8,1.126669398E9,0.6599277680715925,1.1614997050962787E9,3.483030709627867E7,PROJECTED_OVERRUN
410160,해외영업팀,6.73027819E8,9.90791166E8,0.6584200617112159,1.0221860756350876E9,3.139490963508761E7,PROJECTED_OVERRUN
410190,품질관리팀,6.2719069E8,9.13861313E8,0.6638161774747632,9.44825858848317E8,3.0964545848317027E7,PROJECTED_OVERRUN
410230,시설관리팀,5.91744806E8,8.66241198E8,0.6604451910403311,8.959786732156919E8,2.9737475215691924E7,PROJECTED_OVERRUN
410120,정보기술팀,3.87108275E8,5.5190751E8,0.6659794772351031,5.81261567709456E8,2.9354057709455967E7,PROJECTED_OVERRUN
410110,재무회계팀,5.07234876E8,7.39411608E8,0.6605996528083999,7.678400584129859E8,2.842845041298592E7,PROJECTED_OVERRUN
410340,물류팀 2,4.92471552E8,7.22531732E8,0.6572425750089148,7.492995291629124E8,2.676779716291237E7,PROJECTED_OVERRUN
410200,물류팀,7.61830989E8,1.122164749E9,0.6642782570903699,1.1468552234976416E9,2.4690474497641563E7,PROJECTED_OVERRUN


In [0]:
print("=== Projected Overrun Departments ===")

display(
    scenario_base_df

    .filter(
        F.col("forecast_gap") > 0
    )

    .select(
        "cost_center",
        "department_name",
        "actual_ytd",
        "current_budget",
        "seasonal_eac",
        "forecast_gap"
    )

    .orderBy(
        F.desc("forecast_gap")
    )

    .limit(10)
)


print("=== Projected Budget Surplus Departments ===")

display(
    scenario_base_df

    .filter(
        F.col("forecast_gap") < 0
    )

    .select(
        "cost_center",
        "department_name",
        "actual_ytd",
        "current_budget",
        "seasonal_eac",
        "forecast_gap"
    )

    .orderBy(
        "forecast_gap"
    )

    .limit(10)
)

=== Projected Overrun Departments ===


cost_center,department_name,actual_ytd,current_budget,seasonal_eac,forecast_gap
410500,마케팅팀 3,5.55139231E8,7.8911233E8,9.780578335399586E8,1.889455035399586E8
410290,국내영업팀 2,6.90156513E8,9.86795667E8,1.0382254702586175E9,5.142980325861752E7
410470,품질관리팀 3,7.66505908E8,1.126669398E9,1.1614997050962787E9,3.483030709627867E7
410160,해외영업팀,6.73027819E8,9.90791166E8,1.0221860756350876E9,3.139490963508761E7
410190,품질관리팀,6.2719069E8,9.13861313E8,9.44825858848317E8,3.0964545848317027E7
410230,시설관리팀,5.91744806E8,8.66241198E8,8.959786732156919E8,2.9737475215691924E7
410120,정보기술팀,3.87108275E8,5.5190751E8,5.81261567709456E8,2.9354057709455967E7
410110,재무회계팀,5.07234876E8,7.39411608E8,7.678400584129859E8,2.842845041298592E7
410340,물류팀 2,4.92471552E8,7.22531732E8,7.492995291629124E8,2.676779716291237E7
410200,물류팀,7.61830989E8,1.122164749E9,1.1468552234976416E9,2.4690474497641563E7


=== Projected Budget Surplus Departments ===


cost_center,department_name,actual_ytd,current_budget,seasonal_eac,forecast_gap
410400,정보기술팀 3,4.23622868E8,7.82086565E8,6.376295966172419E8,-1.4445696838275814E8
410440,해외영업팀 3,5.36992143E8,9.55299423E8,8.114985257774643E8,-1.4380089722253573E8
410520,경영기획팀 4,3.02009375E8,5.90866238E8,4.532691859018826E8,-1.375970520981174E8
410260,정보기술팀 2,5.46469812E8,8.86574856E8,8.38719440144385E8,-4.785541585561502E7
410250,재무회계팀 2,5.98254583E8,9.51142561E8,9.0543518588939E8,-4.570737511061001E7
410350,고객지원팀 2,4.0146905E8,6.47334482E8,6.041586483584937E8,-4.3175833641506314E7
410450,연구개발팀 3,7.00107583E8,1.081715239E9,1.042239555285201E9,-3.947568371479905E7
410530,재무회계팀 4,5.18355172E8,8.2063015E8,7.842166543758487E8,-3.641349562415135E7
410310,연구개발팀 2,7.30727405E8,1.141869104E9,1.1107581642667768E9,-3.11109397332232E7
410410,인사팀 3,3.57525025E8,5.63960241E8,5.393874614563131E8,-2.4572779543686867E7


In [0]:
# =============================================
# Scenario Input
# =============================================

SOURCE_DEPARTMENT = "재무회계팀 4"
TARGET_DEPARTMENT = "마케팅팀 3"

REALLOCATION_AMOUNT = 100_000_000

In [0]:
# =============================================
# Scenario Calculation
# =============================================

scenario_df = (
    scenario_base_df

    .withColumn(
        "scenario_budget",

        F.when(
            F.col("department_name")
            == SOURCE_DEPARTMENT,

            F.col("current_budget")
            - REALLOCATION_AMOUNT
        )

        .when(
            F.col("department_name")
            == TARGET_DEPARTMENT,

            F.col("current_budget")
            + REALLOCATION_AMOUNT
        )

        .otherwise(
            F.col("current_budget")
        )
    )

    # Scenario 적용 후 예상 Gap
    .withColumn(
        "scenario_forecast_gap",

        F.col("seasonal_eac")
        - F.col("scenario_budget")
    )

    .withColumn(
        "scenario_status",

        F.when(
            F.col("scenario_forecast_gap") > 0,

            "PROJECTED_OVERRUN"
        )

        .otherwise(
            "WITHIN_BUDGET"
        )
    )

    # Budget 조정 효과
    .withColumn(
        "gap_improvement",

        F.col("forecast_gap")
        - F.col("scenario_forecast_gap")
    )
)


display(
    scenario_df

    .filter(
        F.col("department_name").isin(
            SOURCE_DEPARTMENT,
            TARGET_DEPARTMENT
        )
    )

    .select(
        "department_name",

        "current_budget",
        "scenario_budget",

        "seasonal_eac",

        "forecast_gap",
        "scenario_forecast_gap",

        "forecast_status",
        "scenario_status",

        "gap_improvement"
    )
)

department_name,current_budget,scenario_budget,seasonal_eac,forecast_gap,scenario_forecast_gap,forecast_status,scenario_status,gap_improvement
마케팅팀 3,7.8911233E8,8.8911233E8,9.780578335399586E8,1.889455035399586E8,8.89455035399586E7,PROJECTED_OVERRUN,PROJECTED_OVERRUN,1.0E8
재무회계팀 4,8.2063015E8,7.2063015E8,7.842166543758487E8,-3.641349562415135E7,6.358650437584865E7,WITHIN_BUDGET,PROJECTED_OVERRUN,-1.0E8


In [0]:
scenario_summary_df = (
    scenario_df

    .agg(
        F.sum(
            "current_budget"
        ).alias("baseline_total_budget"),

        F.sum(
            "scenario_budget"
        ).alias("scenario_total_budget"),

        F.sum(
            F.when(
                F.col("forecast_gap") > 0,
                F.col("forecast_gap")
            ).otherwise(0)
        ).alias("baseline_total_projected_overrun"),

        F.sum(
            F.when(
                F.col("scenario_forecast_gap") > 0,
                F.col("scenario_forecast_gap")
            ).otherwise(0)
        ).alias("scenario_total_projected_overrun"),

        F.sum(
            F.when(
                F.col("forecast_gap") > 0,
                1
            ).otherwise(0)
        ).alias("baseline_overrun_department_count"),

        F.sum(
            F.when(
                F.col("scenario_forecast_gap") > 0,
                1
            ).otherwise(0)
        ).alias("scenario_overrun_department_count")
    )
)


display(
    scenario_summary_df
)

baseline_total_budget,scenario_total_budget,baseline_total_projected_overrun,scenario_total_projected_overrun,baseline_overrun_department_count,scenario_overrun_department_count
3.672575563E10,3.672575563E10,6.604300832418268E8,6.240165876176753E8,25,26


In [0]:
# =============================================
# Scenario Validation
# =============================================

validation_row = (
    scenario_df
    .agg(
        F.sum("current_budget")
        .alias("baseline_total_budget"),

        F.sum("scenario_budget")
        .alias("scenario_total_budget"),

        F.min("scenario_budget")
        .alias("minimum_scenario_budget")
    )
    .first()
)


baseline_total_budget = (
    validation_row["baseline_total_budget"]
)

scenario_total_budget = (
    validation_row["scenario_total_budget"]
)

minimum_scenario_budget = (
    validation_row["minimum_scenario_budget"]
)


# ---------------------------------------------
# 1. 전체 Budget 보존 확인
# ---------------------------------------------

budget_difference = (
    scenario_total_budget
    - baseline_total_budget
)


# ---------------------------------------------
# 2. Source / Target 결과 확인
# ---------------------------------------------

selected_departments = (
    scenario_df

    .filter(
        F.col("department_name").isin(
            SOURCE_DEPARTMENT,
            TARGET_DEPARTMENT
        )
    )

    .select(
        "department_name",
        "current_budget",
        "scenario_budget",
        "seasonal_eac",
        "forecast_gap",
        "scenario_forecast_gap",
        "forecast_status",
        "scenario_status"
    )

    .orderBy("department_name")
)


print("=== Scenario Validation ===")

print(
    f"Baseline Total Budget : "
    f"{baseline_total_budget:,.0f}"
)

print(
    f"Scenario Total Budget : "
    f"{scenario_total_budget:,.0f}"
)

print(
    f"Budget Difference     : "
    f"{budget_difference:,.0f}"
)

print(
    f"Minimum Scenario Budget : "
    f"{minimum_scenario_budget:,.0f}"
)


if abs(budget_difference) < 1:

    print(
        "PASS - Total Budget is preserved."
    )

else:

    print(
        "FAIL - Total Budget changed."
    )


if minimum_scenario_budget >= 0:

    print(
        "PASS - No negative Scenario Budget."
    )

else:

    print(
        "FAIL - Negative Scenario Budget exists."
    )


display(
    selected_departments
)

=== Scenario Validation ===
Baseline Total Budget : 36,725,755,630
Scenario Total Budget : 36,725,755,630
Budget Difference     : 0
Minimum Scenario Budget : 292,340,152
PASS - Total Budget is preserved.
PASS - No negative Scenario Budget.


department_name,current_budget,scenario_budget,seasonal_eac,forecast_gap,scenario_forecast_gap,forecast_status,scenario_status
마케팅팀 3,7.8911233E8,8.8911233E8,9.780578335399586E8,1.889455035399586E8,8.89455035399586E7,PROJECTED_OVERRUN,PROJECTED_OVERRUN
재무회계팀 4,8.2063015E8,7.2063015E8,7.842166543758487E8,-3.641349562415135E7,6.358650437584865E7,WITHIN_BUDGET,PROJECTED_OVERRUN
